In [ ]:
#| default_exp logger

# logger

> simple logger using idiomatic Solveit

In [ ]:
#| export
import json
from fastcore.all import patch
from datetime import datetime
from dialoghelper.core import update_msg, find_msg_id, read_msg

In [ ]:
import random
import IPython.display
from IPython.display import Markdown
import fastcore.all as FC
from fastcore.test import *
from dutil.core import waitpred

In [ ]:
#| export
class Logger:
    "Timestamped logger that uses a cell's output as sink"
    msgid:str; dname:str
    def __init__(self, id:str='', dname:str='', clear:bool=True): self.setup(id, dname, clear if not dname else False)
    def setup(self, id:str='', dname:str='', clear:bool=False): 
        "Setup logger for current message cell"
        self.dname, self.msgid = dname, id or find_msg_id()
        if clear: self.clear()
        self._s = read_msg(0, id=self.msgid, dname=self.dname).output if dname else getattr(self, '_s', '')
    def clear(self): 
        "Clear all log entries and output"
        self._s=''; update_msg(self.msgid, output='', dname=self.dname)
    @property
    def logs(self): return self._s.splitlines()#read_msg(0, self.msgid).output.splitlines()
    def __str__(self): return self._s#read_msg(0, self.msgid).output
    def __call__(self, msg, *args, **kwargs): 
        "Add timestamped message to log"
        dt = datetime.now(); s = f"[{dt:%H:%M:%S}.{dt.microsecond//1000:03d}] {msg}"
        # msg_insert_line(self.msgid, 0, s, dname=self.dname, update_output=True)  # bug
        if self.dname: self._s = read_msg(0, id=self.msgid, dname=self.dname).output
        self._s = s + (f"\n{self._s}" if self._s != '' else '')
        out = '[{"name": "stdout", "output_type": "stream", "text": %s}]' % json.dumps(self._s)
        update_msg(self.msgid, output=out, dname=self.dname)

In [ ]:
log = Logger()

In [ ]:
log('test')
log('test2')
log('test3')

In [ ]:
print(log)

[19:19:01.817] test3
[19:19:01.791] test2
[19:19:01.765] test


In [ ]:
log

In [ ]:
log(s := ';qwedcv fjkds')
test_is(s in log.logs[0], True)
log.logs

['[19:19:01.861] ;qwedcv fjkds',
 '[19:19:01.817] test3',
 '[19:19:01.791] test2',
 '[19:19:01.765] test']

In [ ]:
log.clear()
test_eq(log.logs, [])

In [ ]:
log(f"Some msg {random.randint(0, 1000)}")
log.logs

['[19:19:02.133] Some msg 93']

In [ ]:
#| export
@patch
def show(self:Logger, clear:bool=False):
    "Display log in current cell, optionally clearing first"
    if not self.dname:
        if self.msgid != find_msg_id():
            oldid = self.msgid
            self.setup()
            update_msg(oldid, output='', dname=self.dname)
        if clear: self.clear()
    print(self._s)

In [ ]:
log.show()

In [ ]:
log.show()

In [ ]:
for _ in "En algún lugar de la Mancha".split(): log(_)

In [ ]:
log.show()

[19:29:11.729] asd
[19:19:02.455] Mancha
[19:19:02.355] la
[19:19:02.329] de
[19:19:02.303] lugar
[19:19:02.278] algún
[19:19:02.252] En
[19:19:02.133] Some msg 93


Look at `show` method. What does it do?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `show` method does three things:

1. **Checks if it's being called from a different cell than where the logger was last set up.** If so, it moves the logger to the new cell (updates `self.msgid` to the current cell) and clears the output from the old cell.

2. **Optionally clears the log** if `clear=True` is passed.

3. **Prints the current log content** (`self._s`) to the current cell's output.

This lets you call `log.show()` from any cell to display the accumulated logs there, and the logger will automatically "move" its display to that new cell while cleaning up the previous one.

Look at the `Logger` class and all the examples. What does it do?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `Logger` class creates a simple logging system that displays timestamped messages in a cell's output area. Here's what it does:

**Core functionality:**
- Logs messages with timestamps (down to milliseconds)
- Stores messages in reverse chronological order (newest first)
- Updates a specific message cell's output in real-time as you add log entries

**Key features:**
- `log('message')` adds a timestamped entry
- `log.clear()` wipes all entries
- `log.show()` displays the log (and handles moving between cells)
- Messages are kept in `log.logs` as a list

**The clever bit:** It uses `update_msg()` to write directly to a cell's output, so you can log from anywhere in your notebook and see all the messages accumulate in one place. Notice how in the examples, multiple `log()` calls build up a running list with timestamps.

What aspect would you like to explore more? The timestamp formatting, how it tracks which cell to update, or something else?

# export -

In [ ]:
from dutil.flakes import show_flakes
show_flakes()

No warnings to report

In [ ]:
# #|hide
# #|eval: false
# import fastcore.all as FC
# from nbdev import nbdev_export
# if FC.IN_NOTEBOOK:
#     nb_path = '01_logger.ipynb'
#     nbdev_export(nb_path)